In [ ]:

import numpy as np
from operator import itemgetter

def conformal(group, G, group_items, profile_weights, top_K):

def conformal(group, G, group_items, weights, top_K ):
    # Build V_set (all candidate items from users in the group)
    V_set = set().union(*(V[user] for user in group))

    # Profile = items not in V_set
    profile = list(set(profile_wt.keys()) - V_set)
    n = len(profile)

    # Precompute weights/support for profile items
    weights = np.array([profile_wt[item] for item in profile])
    supports = np.array([support[item] for item in profile])

    # Build pp (list of sorted contribution vectors)
    pp = []
    for v in V_set:
        coexist = coexistenceCount[profile, v]
        ppi = weights * coexist / supports
        ppi.sort()                # in-place, faster than np.sort
        pp.append(ppi)

    # Scaling factor
    avg_r = np.mean(list(rec_candidates.values()))
    avg_p = np.mean(list(profile_wt.values()))
    scale = avg_p / avg_r

    # Recommendations
    recommendations = {}
    pp = np.array(pp, dtype=object)  # keep as object if different lengths

    for r, r_val in rec_candidates.items():
        scores = []
        for ppi, v in zip(pp, V_set):
            pp_nv = scale * r_val * coexistenceCount[r, v] / support[r]
            p = np.searchsorted(ppi, pp_nv, side="right")
            scores.append(p / n)
        recommendations[r] = np.mean(scores)

    # Sort once
    recommendations_sorted = dict(sorted(recommendations.items(),
                                         key=itemgetter(1), reverse=True))

    recommendations_topK = dict(list(recommendations_sorted.items())[:top_K])

    return recommendations_topK, recommendations_sorted
